# 01 - Analise Exploratoria do Dataset NIH Chest X-rays

Este notebook executa a EDA do dataset NIH Chest X-rays para apoiar as decisoes tecnicas do PBL CardioIA Vision.

Objetivos:

1. Ler `Data_entry_2017.csv`.
2. Validar colunas, nulos e duplicatas.
3. Analisar distribuicao das labels.
4. Medir frequencia de `No Finding` e `Cardiomegaly`.
5. Avaliar casos multi-label.
6. Analisar pacientes, sexo, idade e posicao da imagem.
7. Verificar se as imagens existem localmente via `image_paths.csv`.
8. Salvar graficos e tabelas para o relatorio.

Esta EDA sera usada para justificar balanceamento, splits por paciente e escolha inicial do problema binario `No Finding` vs `Cardiomegaly`.

## Pre-requisito

Execute antes o notebook:

`notebooks/00_download_dataset_kaggle.ipynb`

Ele deve gerar, pelo menos:

```text
data/raw/Data_entry_2017.csv
data/raw/image_paths.csv
data/raw/images/
```

Se o dataset ainda nao foi baixado, este notebook vai falhar com uma mensagem clara indicando a etapa anterior.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

config.ensure_project_directories()

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.axisbelow"] = True

if HAS_SEABORN:
    sns.set_theme(style="whitegrid")

FIGURES_DIR = config.FIGURES_DIR
TABLES_DIR = config.TABLES_DIR
RAW_DIR = config.RAW_DATA_DIR
CSV_PATH = RAW_DIR / config.DATA_ENTRY_FILENAME
IMAGE_INDEX_PATH = RAW_DIR / config.IMAGE_PATHS_FILENAME

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CSV_PATH:", CSV_PATH)
print("IMAGE_INDEX_PATH:", IMAGE_INDEX_PATH)
print("FIGURES_DIR:", FIGURES_DIR)
print("TABLES_DIR:", TABLES_DIR)

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo nao encontrado: {CSV_PATH}. "
        "Execute primeiro notebooks/00_download_dataset_kaggle.ipynb para baixar e organizar o dataset."
    )

df = pd.read_csv(CSV_PATH)

print("Linhas:", len(df))
print("Colunas:", len(df.columns))
df.head()

## Validacao Inicial do CSV

Nesta etapa verificamos se as colunas principais existem, se ha valores nulos e se ha duplicatas por imagem.

In [ ]:
required_columns = [
    "Image Index",
    "Finding Labels",
    "Patient ID",
    "Patient Age",
    "Patient Gender",
    "View Position",
]

missing_required_columns = [column for column in required_columns if column not in df.columns]
if missing_required_columns:
    raise ValueError(f"Colunas obrigatorias ausentes: {missing_required_columns}")

null_counts = df[required_columns].isna().sum().sort_values(ascending=False)
duplicate_image_rows = int(df.duplicated(subset=["Image Index"]).sum())
unique_patients = int(df["Patient ID"].nunique())

validation_table = pd.DataFrame(
    {
        "metric": [
            "total_rows",
            "total_columns",
            "unique_images",
            "duplicate_image_rows",
            "unique_patients",
        ],
        "value": [
            len(df),
            len(df.columns),
            df["Image Index"].nunique(),
            duplicate_image_rows,
            unique_patients,
        ],
    }
)

validation_table.to_csv(TABLES_DIR / "eda_validacao_inicial.csv", index=False)
null_counts.to_csv(TABLES_DIR / "eda_nulos_colunas_principais.csv", header=["null_count"])

display(validation_table)
display(null_counts.to_frame("null_count"))

## Preparacao das Labels

O NIH Chest X-rays e multi-label: algumas imagens possuem mais de uma condicao separada por `|`. Por isso, criamos colunas auxiliares para contar labels e identificar casos com `Cardiomegaly`.

In [ ]:
df = df.copy()
df["labels_list"] = df["Finding Labels"].astype(str).str.split("|")
df["num_labels"] = df["labels_list"].apply(len)
df["is_no_finding"] = df["Finding Labels"].eq(config.NEGATIVE_LABEL)
df["has_cardiomegaly"] = df["labels_list"].apply(lambda labels: config.TARGET_LABEL in labels)
df["is_cardiomegaly_only"] = df["Finding Labels"].eq(config.TARGET_LABEL)
df["is_multilabel"] = df["num_labels"] > 1

df_labels_exploded = df[["Image Index", "Patient ID", "labels_list"]].explode("labels_list")
df_labels_exploded = df_labels_exploded.rename(columns={"labels_list": "label"})

label_distribution = (
    df_labels_exploded["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="image_count")
)
label_distribution["percentage_of_rows"] = label_distribution["image_count"] / len(df) * 100
label_distribution.to_csv(TABLES_DIR / "eda_distribuicao_labels.csv", index=False)

label_distribution

## Distribuicao das Labels

Este grafico mostra quais achados aparecem mais no dataset. Isso e importante para demonstrar desbalanceamento.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
plot_data = label_distribution.sort_values("image_count", ascending=True)

if HAS_SEABORN:
    sns.barplot(data=plot_data, x="image_count", y="label", ax=ax, color="#4C78A8")
else:
    ax.barh(plot_data["label"], plot_data["image_count"], color="#4C78A8")

ax.set_title("Distribuicao de labels no NIH Chest X-rays")
ax.set_xlabel("Quantidade de imagens")
ax.set_ylabel("Label")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "eda_distribuicao_labels.png", dpi=160, bbox_inches="tight")
plt.show()

## Foco do Projeto: No Finding vs Cardiomegaly

O problema inicial recomendado e binario:

- Classe 0: `No Finding`
- Classe 1: `Cardiomegaly`

Aqui medimos o tamanho potencial das duas classes e se `Cardiomegaly` aparece isolada ou acompanhada de outras patologias.

In [ ]:
binary_focus_summary = pd.DataFrame(
    {
        "metric": [
            "no_finding_exact",
            "has_cardiomegaly_any_label",
            "cardiomegaly_only",
            "cardiomegaly_with_other_findings",
            "multilabel_rows_total",
        ],
        "image_count": [
            int(df["is_no_finding"].sum()),
            int(df["has_cardiomegaly"].sum()),
            int(df["is_cardiomegaly_only"].sum()),
            int((df["has_cardiomegaly"] & df["is_multilabel"]).sum()),
            int(df["is_multilabel"].sum()),
        ],
    }
)
binary_focus_summary["percentage_of_rows"] = binary_focus_summary["image_count"] / len(df) * 100
binary_focus_summary.to_csv(TABLES_DIR / "eda_resumo_no_finding_cardiomegaly.csv", index=False)
binary_focus_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_data = binary_focus_summary.copy()
plot_data["metric"] = plot_data["metric"].str.replace("_", " ")

if HAS_SEABORN:
    sns.barplot(data=plot_data, x="metric", y="image_count", ax=ax, color="#59A14F")
else:
    ax.bar(plot_data["metric"], plot_data["image_count"], color="#59A14F")

ax.set_title("Resumo do recorte No Finding vs Cardiomegaly")
ax.set_xlabel("Grupo")
ax.set_ylabel("Quantidade de imagens")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "eda_no_finding_vs_cardiomegaly.png", dpi=160, bbox_inches="tight")
plt.show()

## Analise de Multi-label

Como o dataset tem imagens com multiplos achados, precisamos medir quantas imagens possuem uma ou mais labels. Isso influencia a escolha entre dataset binario limpo e dataset binario realista.

In [ ]:
num_labels_distribution = (
    df["num_labels"]
    .value_counts()
    .sort_index()
    .rename_axis("num_labels")
    .reset_index(name="image_count")
)
num_labels_distribution["percentage_of_rows"] = num_labels_distribution["image_count"] / len(df) * 100
num_labels_distribution.to_csv(TABLES_DIR / "eda_distribuicao_quantidade_labels_por_imagem.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(num_labels_distribution["num_labels"], num_labels_distribution["image_count"], color="#F28E2B")
ax.set_title("Quantidade de labels por imagem")
ax.set_xlabel("Numero de labels na imagem")
ax.set_ylabel("Quantidade de imagens")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "eda_quantidade_labels_por_imagem.png", dpi=160, bbox_inches="tight")
plt.show()

num_labels_distribution

## Distribuicao por Sexo, Idade e Posicao da Imagem

Essas variaveis serao importantes para a discussao de governanca e fairness. Nesta etapa ainda nao medimos desempenho por subgrupo, mas ja entendemos a representatividade do dataset.

In [ ]:
gender_distribution = (
    df["Patient Gender"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("patient_gender")
    .reset_index(name="image_count")
)
gender_distribution["percentage_of_rows"] = gender_distribution["image_count"] / len(df) * 100
gender_distribution.to_csv(TABLES_DIR / "eda_distribuicao_sexo.csv", index=False)

view_distribution = (
    df["View Position"]
    .fillna("Unknown")
    .value_counts()
    .rename_axis("view_position")
    .reset_index(name="image_count")
)
view_distribution["percentage_of_rows"] = view_distribution["image_count"] / len(df) * 100
view_distribution.to_csv(TABLES_DIR / "eda_distribuicao_posicao_imagem.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(gender_distribution["patient_gender"], gender_distribution["image_count"], color="#B07AA1")
axes[0].set_title("Distribuicao por sexo")
axes[0].set_xlabel("Sexo")
axes[0].set_ylabel("Quantidade de imagens")

axes[1].bar(view_distribution["view_position"], view_distribution["image_count"], color="#E15759")
axes[1].set_title("Distribuicao por posicao da imagem")
axes[1].set_xlabel("Posicao")
axes[1].set_ylabel("Quantidade de imagens")

plt.tight_layout()
fig.savefig(FIGURES_DIR / "eda_distribuicao_sexo_posicao.png", dpi=160, bbox_inches="tight")
plt.show()

display(gender_distribution)
display(view_distribution)

In [ ]:
age_series = pd.to_numeric(df["Patient Age"], errors="coerce")
age_summary = age_series.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("patient_age")
age_summary.to_csv(TABLES_DIR / "eda_resumo_idade.csv")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(age_series.dropna(), bins=40, color="#76B7B2", edgecolor="white")
ax.set_title("Distribuicao de idade dos pacientes")
ax.set_xlabel("Idade")
ax.set_ylabel("Quantidade de imagens")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "eda_distribuicao_idade.png", dpi=160, bbox_inches="tight")
plt.show()

age_summary

## Relacao entre Cardiomegaly e Subgrupos

Aqui medimos a proporcao de imagens com `Cardiomegaly` por sexo e posicao. Isso ainda nao prova vies do modelo, mas aponta variaveis que devem ser acompanhadas depois na analise de fairness.

In [ ]:
def subgroup_rate_table(data: pd.DataFrame, group_col: str) -> pd.DataFrame:
    grouped = (
        data.groupby(group_col, dropna=False)
        .agg(
            image_count=("Image Index", "count"),
            cardiomegaly_count=("has_cardiomegaly", "sum"),
            no_finding_count=("is_no_finding", "sum"),
        )
        .reset_index()
    )
    grouped["cardiomegaly_rate"] = grouped["cardiomegaly_count"] / grouped["image_count"]
    grouped["no_finding_rate"] = grouped["no_finding_count"] / grouped["image_count"]
    return grouped.sort_values("image_count", ascending=False)


gender_rates = subgroup_rate_table(df, "Patient Gender")
view_rates = subgroup_rate_table(df, "View Position")

gender_rates.to_csv(TABLES_DIR / "eda_taxas_por_sexo.csv", index=False)
view_rates.to_csv(TABLES_DIR / "eda_taxas_por_posicao.csv", index=False)

display(gender_rates)
display(view_rates)

## Validacao do Indice de Imagens

Se `image_paths.csv` existir, verificamos quantas imagens do CSV de metadados foram localizadas. Essa validacao conecta a EDA ao pipeline de pre-processamento.

In [ ]:
if IMAGE_INDEX_PATH.exists():
    image_index = pd.read_csv(IMAGE_INDEX_PATH)
    indexed_images = set(image_index["image_name"].astype(str))
    metadata_images = set(df["Image Index"].astype(str))
    found_images = metadata_images & indexed_images
    missing_images = metadata_images - indexed_images
    image_validation_summary = pd.DataFrame(
        {
            "metric": [
                "metadata_unique_images",
                "indexed_local_images",
                "metadata_images_found_locally",
                "metadata_images_missing_locally",
            ],
            "value": [
                len(metadata_images),
                len(indexed_images),
                len(found_images),
                len(missing_images),
            ],
        }
    )
else:
    image_index = pd.DataFrame(columns=["image_name", "absolute_path", "relative_path"])
    found_images = set()
    missing_images = set(df["Image Index"].astype(str))
    image_validation_summary = pd.DataFrame(
        {
            "metric": ["image_paths_csv_found", "metadata_images_missing_locally"],
            "value": [0, len(missing_images)],
        }
    )

image_validation_summary.to_csv(TABLES_DIR / "eda_validacao_imagens.csv", index=False)
image_validation_summary

## Visualizacao de Amostras

As amostras ajudam a conferir se o pipeline de imagens esta funcionando. Se `image_paths.csv` ainda nao existir, esta celula apenas informa que a visualizacao depende do notebook de download.

In [ ]:
def plot_image_samples(sample_df: pd.DataFrame, title: str, output_name: str, max_images: int = 6) -> None:
    if image_index.empty or "absolute_path" not in image_index.columns:
        print(f"Sem image_paths.csv valido para visualizar amostras de {title}.")
        return

    path_lookup = dict(zip(image_index["image_name"].astype(str), image_index["absolute_path"].astype(str)))
    sample_names = [name for name in sample_df["Image Index"].astype(str).tolist() if name in path_lookup][:max_images]

    if not sample_names:
        print(f"Nenhuma imagem local encontrada para {title}.")
        return

    cols = 3
    rows = int(np.ceil(len(sample_names) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, image_name in zip(axes, sample_names):
        image_path = Path(path_lookup[image_name])
        image = Image.open(image_path).convert("L")
        label = df.loc[df["Image Index"] == image_name, "Finding Labels"].iloc[0]
        ax.imshow(image, cmap="gray")
        ax.set_title(f"{image_name}\n{label}", fontsize=9)
        ax.axis("off")

    for ax in axes[len(sample_names):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / output_name, dpi=160, bbox_inches="tight")
    plt.show()


plot_image_samples(df[df["is_no_finding"]].head(30), "Amostras No Finding", "eda_amostras_no_finding.png")
plot_image_samples(df[df["has_cardiomegaly"]].head(30), "Amostras com Cardiomegaly", "eda_amostras_cardiomegaly.png")

## Conclusoes da EDA

A celula abaixo gera um resumo objetivo para apoiar o relatorio tecnico e a proxima etapa de pre-processamento.

In [ ]:
total_rows = len(df)
no_finding_count = int(df["is_no_finding"].sum())
cardiomegaly_any_count = int(df["has_cardiomegaly"].sum())
cardiomegaly_only_count = int(df["is_cardiomegaly_only"].sum())
multilabel_count = int(df["is_multilabel"].sum())

eda_conclusions = [
    {
        "question": "O dataset e balanceado?",
        "answer": (
            "Nao. A distribuicao de labels mostra diferencas grandes entre classes. "
            "Isso justifica balanceamento, class weights ou weighted sampler nas proximas etapas."
        ),
    },
    {
        "question": "Quantas imagens No Finding existem?",
        "answer": f"{no_finding_count:,} imagens ({no_finding_count / total_rows:.2%} das linhas).",
    },
    {
        "question": "Quantas imagens contem Cardiomegaly?",
        "answer": (
            f"{cardiomegaly_any_count:,} imagens contem Cardiomegaly em qualquer posicao de label; "
            f"{cardiomegaly_only_count:,} imagens possuem apenas Cardiomegaly."
        ),
    },
    {
        "question": "Existem muitos casos multi-label?",
        "answer": f"{multilabel_count:,} imagens possuem mais de uma label ({multilabel_count / total_rows:.2%}).",
    },
    {
        "question": "Ha diferenca relevante entre PA e AP?",
        "answer": "Ver tabela eda_distribuicao_posicao_imagem.csv e eda_taxas_por_posicao.csv para avaliar distribuicao e taxa de Cardiomegaly por posicao.",
    },
    {
        "question": "Ha possivel vies por idade ou sexo?",
        "answer": "A EDA registra distribuicoes por idade e sexo. A conclusao de vies do modelo deve ser feita depois, usando metricas por subgrupo no conjunto de teste.",
    },
]

eda_conclusions_df = pd.DataFrame(eda_conclusions)
eda_conclusions_df.to_csv(TABLES_DIR / "eda_conclusoes.csv", index=False)
eda_conclusions_df

## Arquivos Gerados

Tabelas principais:

- `reports/tabelas/eda_validacao_inicial.csv`
- `reports/tabelas/eda_nulos_colunas_principais.csv`
- `reports/tabelas/eda_distribuicao_labels.csv`
- `reports/tabelas/eda_resumo_no_finding_cardiomegaly.csv`
- `reports/tabelas/eda_distribuicao_quantidade_labels_por_imagem.csv`
- `reports/tabelas/eda_distribuicao_sexo.csv`
- `reports/tabelas/eda_distribuicao_posicao_imagem.csv`
- `reports/tabelas/eda_resumo_idade.csv`
- `reports/tabelas/eda_taxas_por_sexo.csv`
- `reports/tabelas/eda_taxas_por_posicao.csv`
- `reports/tabelas/eda_validacao_imagens.csv`
- `reports/tabelas/eda_conclusoes.csv`

Figuras principais:

- `reports/figuras/eda_distribuicao_labels.png`
- `reports/figuras/eda_no_finding_vs_cardiomegaly.png`
- `reports/figuras/eda_quantidade_labels_por_imagem.png`
- `reports/figuras/eda_distribuicao_sexo_posicao.png`
- `reports/figuras/eda_distribuicao_idade.png`
- `reports/figuras/eda_amostras_no_finding.png`
- `reports/figuras/eda_amostras_cardiomegaly.png`

## Proxima Etapa

Depois desta EDA, seguir para as tarefas 6, 7 e 8:

1. Criar dataset binario limpo e realista.
2. Implementar balanceamento.
3. Criar splits por paciente.

Essas decisoes devem usar os resultados desta EDA, principalmente `No Finding`, `Cardiomegaly`, multi-label e distribuicao por paciente.